In [3]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [4]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Wang2021_Part2.h5ad")

In [5]:
adata = adata[adata.obs['donor_id'].isin(["C3L-02705", "C3L-03405", "C3L-03968", "C3N-00662", "C3N-01334", "C3N-01798", "C3N-01814"])]

In [6]:
df_obs = pd.DataFrame(adata.obs)

In [7]:
del adata.obs

In [8]:
adata = adata.raw.to_adata()

In [9]:
adata

AnnData object with n_obs × n_vars = 51418 × 19254
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'development_stage_colors', 'donor_id_colors', 'gbmap_colors', 'hvg', 'is_primary_data_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'stage_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [10]:
#adata = adata.raw.to_adata()

In [11]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 1418/1418 [00:02<00:00, 687.88it/s]


In [12]:
adata.X = X_counts_recovered

In [13]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [14]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [15]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [16]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [17]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,10514,6.956510,True,0.060899,0.061489,1.054076
SULT1B1,False,640,0.423451,True,0.003422,0.003355,1.066482
TRIM63,False,123,0.081382,True,0.000734,0.000857,1.293688
HDHD2,False,5243,3.468992,True,0.023329,0.019382,0.873629
MORF4L2-AS1,False,1518,1.004373,True,0.006681,0.005571,0.879117
...,...,...,...,...,...,...,...
LINC01916,False,79,0.052270,True,0.000383,0.000350,0.989575
LINC02360,False,295,0.195185,True,0.001196,0.000947,0.816293
LINC02096,False,82,0.054255,True,0.000409,0.000403,1.058072


In [18]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [19]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [20]:
adata.var = df_tmp

In [21]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [22]:
adata

View of AnnData object with n_obs × n_vars = 51418 × 17710
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'development_stage_colors', 'donor_id_colors', 'gbmap_colors', 'hvg', 'is_primary_data_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'stage_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [23]:
adata.obs['donor_id'] = df_obs['donor_id']

In [24]:
metadata_data = {
    'Author': ['Wang2021'] * 7,
    'donor_id': ["C3L-02705", "C3L-03405", "C3L-03968", "C3N-00662", "C3N-01334", "C3N-01798", "C3N-01814"],
    'stage': ['Primary'] * 7,
    'assay': ['10x 3\' v3'] * 7,
    'tissue': ['frontal lobe', 'brain', 'parietal lobe', 'temporal lobe', 'occipital lobe', 'frontal lobe', 'brain'],
    'Cells': ['Total'] * 7,
    'Method': ['nuclei'] * 7
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

     Author   donor_id    stage      assay          tissue  Cells  Method
0  Wang2021  C3L-02705  Primary  10x 3' v3    frontal lobe  Total  nuclei
1  Wang2021  C3L-03405  Primary  10x 3' v3           brain  Total  nuclei
2  Wang2021  C3L-03968  Primary  10x 3' v3   parietal lobe  Total  nuclei
3  Wang2021  C3N-00662  Primary  10x 3' v3   temporal lobe  Total  nuclei
4  Wang2021  C3N-01334  Primary  10x 3' v3  occipital lobe  Total  nuclei
5  Wang2021  C3N-01798  Primary  10x 3' v3    frontal lobe  Total  nuclei
6  Wang2021  C3N-01814  Primary  10x 3' v3           brain  Total  nuclei


In [25]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

        donor_id    Author    stage      assay        tissue  Cells  Method
0      C3L-02705  Wang2021  Primary  10x 3' v3  frontal lobe  Total  nuclei
1      C3L-02705  Wang2021  Primary  10x 3' v3  frontal lobe  Total  nuclei
2      C3L-02705  Wang2021  Primary  10x 3' v3  frontal lobe  Total  nuclei
3      C3L-02705  Wang2021  Primary  10x 3' v3  frontal lobe  Total  nuclei
4      C3L-02705  Wang2021  Primary  10x 3' v3  frontal lobe  Total  nuclei
...          ...       ...      ...        ...           ...    ...     ...
51413  C3N-01814  Wang2021  Primary  10x 3' v3         brain  Total  nuclei
51414  C3N-01814  Wang2021  Primary  10x 3' v3         brain  Total  nuclei
51415  C3N-01814  Wang2021  Primary  10x 3' v3         brain  Total  nuclei
51416  C3N-01814  Wang2021  Primary  10x 3' v3         brain  Total  nuclei
51417  C3N-01814  Wang2021  Primary  10x 3' v3         brain  Total  nuclei

[51418 rows x 7 columns]


In [26]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [27]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
C3L-02705_AAACCCACATATAGCC-1-2-1,C3L-02705,1499,1892.982910,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Interneurons,malignant cell
C3L-02705_AAACCCAGTAACCAGG-1-2-1,C3L-02705,1135,1740.172241,Neoplastic,Differentiated-like,AC-like,Cancer cell,Adipocyte Progenitor Cells,malignant cell
C3L-02705_AAACCCAGTGACTCTA-1-2-1,C3L-02705,2043,1926.726807,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
C3L-02705_AAACCCATCATTGCGA-1-2-1,C3L-02705,3229,2328.941162,Neoplastic,Differentiated-like,MES-like,Astrocyte,Podocytes,malignant cell
C3L-02705_AAACCCATCCAACACA-1-2-1,C3L-02705,1695,2015.467896,Non-neoplastic,Myeloid,TAM-MG,Microglial cell,Macrophages,microglial cell
...,...,...,...,...,...,...,...,...,...
C3N-01814_TTTGTTGTCGACGCTG-1-2-1,C3N-01814,3446,2317.323486,Non-neoplastic,Glial-Neuronal,OPC,Neuron,Pluripotent Stem Cells,oligodendrocyte precursor cell
C3N-01814_TTTGTTGTCGAGAAGC-1-2-1,C3N-01814,2638,2151.104004,Non-neoplastic,Glial-Neuronal,OPC,Oligodendrocyte,Interneurons,oligodendrocyte precursor cell
C3N-01814_TTTGTTGTCGCTTTAT-1-2-1,C3N-01814,4375,2365.847656,Non-neoplastic,Glial-Neuronal,OPC,Neuron,Pluripotent Stem Cells,oligodendrocyte precursor cell
C3N-01814_TTTGTTGTCGTTGTGA-1-2-1,C3N-01814,2302,2052.934082,Neoplastic,Stem-like,OPC-like,Oligodendrocyte,Interneurons,malignant cell


In [28]:
merged_obs_df.index= df_obs.index

In [29]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [30]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [31]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [32]:
del merged_obs_df['donor_id_y']

In [33]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [34]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [35]:
adata.obs = merged_obs_df

In [36]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    600 total control genes are used. (0:00:02)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    725 total control genes are used. (0:00:02)
-->     'phase', cell cycle phase (adata.obs)


In [37]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Wang2021_Part3.h5ad")